<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Day 5 · Gold and AI/BI expectations</h1><p>Independent source-derived calculations; not a Lakehouse engine.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اليوم الخامس · النتائج المرجعية لـGold وAI وBI</h1><p>حسابات مستقلة مشتقة من المصدر؛ ليست تشغيلًا لمحرك Lakehouse.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Goal</h2><p>Derive the final table contracts, reconcile the same 75 trips, and distinguish unknown future labels from zero. <a href="README.md">Day 5 guide</a>.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الهدف</h2><p>اشتقاق مخرجات المشروع النهائية وتسوية الرحلات الـ75 نفسها والتمييز بين النتائج المستقبلية المجهولة والصفر. <a href="README.md">دليل اليوم الخامس</a>.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Verify inputs</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>١. تحقق من المدخلات</h2></td></tr></tbody></table>

In [1]:
from pathlib import Path
import sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the complete course repository")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.workspace import require_fixed_dataset
require_fixed_dataset(SOURCE)
print("MASAR_SMALL_V1: original source hashes verified")

MASAR_SMALL_V1: original source hashes verified


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>2. Build reference tables</h2><p>The same corrected trips feed all consumers; no replacement dataset is introduced.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>٢. ابن الجداول المرجعية</h2><p>تغذي الرحلات المصححة نفسها جميع المخرجات، دون مجموعة بيانات بديلة.</p></td></tr></tbody></table>

In [2]:
from masar.serving_reference import day05_reference
reference = day05_reference(SOURCE)
print("Scope:", reference["scope"])
for table, count in reference["row_counts"].items():
    print(f"{table}: {count} rows")
assert reference["engine_executed"] is False

Scope: DAY05_SOURCE_EXPECTATIONS_ONLY
gold.zone_hourly_demand: 75 rows
gold.driver_daily: 18 rows
bi.dim_zone: 3 rows
bi.dim_driver: 6 rows
bi.dim_date: 3 rows
bi.fact_trips: 75 rows
ai.zone_hourly_features: 3 rows
ai.zone_hourly_labels: 3 rows


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>3. Read the Gold grain</h2><p>Hourly rows count observed trip starts. Missing hours are not certified zero demand. Driver-day rows use the local start date, including the late trips crossing midnight.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>٣. اقرأ مستوى الصف في Gold</h2><p>نعد بدايات الرحلات المرصودة في الساعة؛ الساعات الغائبة ليست طلبًا صفريًا مؤكدًا. يستخدم ملخص السائق تاريخ البداية المحلي حتى للرحلات التي عبرت منتصف الليل.</p></td></tr></tbody></table>

In [3]:
tables = reference["tables"]
print(json.dumps(tables["gold.zone_hourly_demand"][:3], indent=2))
print(json.dumps(tables["gold.driver_daily"][:3], indent=2))

[
  {
    "zone_key": "Z_DAMMAM",
    "hour_utc": "2026-06-01T03:00:00Z",
    "hour_local": "2026-06-01T06:00:00+03:00",
    "trip_count": 1,
    "total_fare_sar": "21.60",
    "total_duration_seconds": 720
  },
  {
    "zone_key": "Z_DAMMAM",
    "hour_utc": "2026-06-01T05:00:00Z",
    "hour_local": "2026-06-01T08:00:00+03:00",
    "trip_count": 1,
    "total_fare_sar": "22.85",
    "total_duration_seconds": 900
  },
  {
    "zone_key": "Z_DAMMAM",
    "hour_utc": "2026-06-01T07:00:00Z",
    "hour_local": "2026-06-01T10:00:00+03:00",
    "trip_count": 1,
    "total_fare_sar": "24.10",
    "total_duration_seconds": 1080
  }
]
[
  {
    "driver_key": "SYN_D001",
    "trip_date_local": "2026-06-01",
    "trip_count": 5,
    "total_fare_sar": "117.00",
    "total_duration_seconds": 4860
  },
  {
    "driver_key": "SYN_D001",
    "trip_date_local": "2026-06-02",
    "trip_count": 4,
    "total_fare_sar": "90.00",
    "total_duration_seconds": 4320
  },
  {
    "driver_key": "SYN_D001",
   

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>4. Reconcile BI independently</h2><p>The SQL check uses integer minor units to avoid floating-point money error. SQLite here is a read-only arithmetic cross-check, not the course storage engine.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>٤. سوّ مخرجات BI بصورة مستقلة</h2><p>يستخدم فحص SQL الهللات الصحيحة لتجنب خطأ التقريب. SQLite هنا أداة تحقق حسابي مستقل، وليس محرك التخزين في الدورة.</p></td></tr></tbody></table>

In [4]:
from masar.serving_reference import independent_sql_check
print(json.dumps(independent_sql_check(tables), indent=2))
assert reference["reconciliation"]["trip_count"] == 75
assert reference["reconciliation"]["fare_minor"] == 188060
assert reference["reconciliation"]["gps_events"] == 217

{
  "scope": "INDEPENDENT_SQL_RECONCILIATION_NOT_SPARK",
  "trip_count": 75,
  "fare_minor": 188060,
  "duration_seconds": 95400,
  "gps_events": 217,
  "zone_totals": [
    {
      "zone_key": "Z_DAMMAM",
      "trip_count": 25,
      "fare_minor": 67040
    },
    {
      "zone_key": "Z_JEDDAH",
      "trip_count": 25,
      "fare_minor": 62520
    },
    {
      "zone_key": "Z_RIYADH",
      "trip_count": 25,
      "fare_minor": 58500
    }
  ],
  "daily_group_reconciliation": true
}


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>5. Inspect time-safe feature inputs</h2><p>At scenario time 4 June 2026, 06:05 Riyadh time, use completed and already delivered history only. The next forecast hour starts at 07:00 Riyadh time.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>٥. افحص خصائص التنبؤ وفق وقت الإتاحة</h2><p>وقت السيناريو هو ٤ يونيو ٢٠٢٦، الساعة ٠٦:٠٥ بتوقيت الرياض. نستخدم الرحلات المنتهية والواصلة بالفعل فقط؛ وتبدأ ساعة التنبؤ التالية عند ٠٧:٠٠.</p></td></tr></tbody></table>

In [5]:
from masar.serving_reference import FEATURE_INPUTS, validate_feature_inputs
validate_feature_inputs(FEATURE_INPUTS)
print("Model-input allowlist:", FEATURE_INPUTS)
print(json.dumps(tables["ai.zone_hourly_features"], indent=2))
print(json.dumps(tables["ai.zone_hourly_labels"], indent=2))
assert all(row["target_trip_count"] is None for row in tables["ai.zone_hourly_labels"])

Model-input allowlist: ('zone_key', 'completed_trips_24h', 'avg_duration_seconds_24h', 'history_available')
[
  {
    "zone_key": "Z_DAMMAM",
    "as_of_utc": "2026-06-04T03:05:00Z",
    "prediction_hour_utc": "2026-06-04T04:00:00Z",
    "history_window_start_utc": "2026-06-03T03:05:00Z",
    "completed_trips_24h": 8,
    "avg_duration_seconds_24h": "1470.00",
    "history_available": true,
    "max_source_available_at_utc": "2026-06-04T03:00:00Z"
  },
  {
    "zone_key": "Z_JEDDAH",
    "as_of_utc": "2026-06-04T03:05:00Z",
    "prediction_hour_utc": "2026-06-04T04:00:00Z",
    "history_window_start_utc": "2026-06-03T03:05:00Z",
    "completed_trips_24h": 8,
    "avg_duration_seconds_24h": "1350.00",
    "history_available": true,
    "max_source_available_at_utc": "2026-06-04T03:00:00Z"
  },
  {
    "zone_key": "Z_RIYADH",
    "as_of_utc": "2026-06-04T03:05:00Z",
    "prediction_hour_utc": "2026-06-04T04:00:00Z",
    "history_window_start_utc": "2026-06-03T03:05:00Z",
    "completed_t

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>6. Check the arrival-time trap</h2><p>These files were delivered on 4 June in the scenario. Looking back to 2 June does not make the later delivery available earlier. An empty observed history is not proof of zero real activity.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>٦. اختبر الفرق بين وقت الحدث ووقت وصوله</h2><p>وصلت الملفات في السيناريو يوم ٤ يونيو. الرجوع إلى ٢ يونيو لا يجعلها متاحة قبل وصولها. غياب التاريخ المتاح لا يثبت غياب النشاط الحقيقي.</p></td></tr></tbody></table>

In [6]:
from masar.delta_reference import day03_reference
from masar.trust_reference import events_from_source, GPS_FILES
from masar.serving_reference import tables_from_trusted
trips = day03_reference(SOURCE)["expected_corrected_rows"]
events = [event for name in GPS_FILES for event in events_from_source(SOURCE, name)]
early = tables_from_trusted(trips, events, SOURCE, as_of="2026-06-02T03:05:00Z")
assert all(not r["history_available"] for r in early["ai.zone_hourly_features"])
print("Early-cutoff history is unavailable for all three city proxies, not certified zero demand.")

Early-cutoff history is unavailable for all three city proxies, not certified zero demand.


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>7. Test two deliberately invalid outputs</h2><p>Transient copies only: the original sources are never edited.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>٧. اختبر مخرجين معيبين عمدًا</h2><p>نستخدم نسخًا مؤقتة فقط دون تعديل المصادر الأصلية.</p></td></tr></tbody></table>

In [7]:
from copy import deepcopy
from masar.serving_reference import validate_tables
for problem in ["duplicate_fact", "invented_zero_label"]:
    probe = deepcopy(tables)
    if problem == "duplicate_fact":
        probe["bi.fact_trips"].append(deepcopy(probe["bi.fact_trips"][0]))
    else:
        probe["ai.zone_hourly_labels"][0]["target_trip_count"] = 0
    try:
        validate_tables(probe)
    except ValueError as exc:
        print(problem, "REJECTED:", exc)
    else:
        raise AssertionError("Deliberate defect was incorrectly accepted")

duplicate_fact REJECTED: Missing or duplicate serving key: bi.fact_trips
invented_zero_label REJECTED: This fixed fixture has no future ground truth; do not invent zero targets


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>8. Save the actual reference result</h2><p>Compare these expectations to native readback in Labs 07/08. They cannot replace native execution evidence.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>٨. احفظ النتيجة المرجعية الفعلية</h2><p>قارن التوقعات بنتائج القراءة الفعلية من المحرك في اللابين ٠٧ و٠٨؛ ولا تعدّها بديلًا عن دليل التشغيل.</p></td></tr></tbody></table>

In [8]:
from masar.workspace import new_workspace, workspace_path, write_json
assert all(reference["checks"].values())
work = new_workspace(ROOT, "day05_reference")
write_json(workspace_path(work, "reports/day05_reference.json"), reference)
print("Reference checks passed:", len(reference["checks"]))
print("New outputs/day05_reference_* workspace saved. No native engine execution claimed.")

Reference checks passed: 18
New outputs/day05_reference_* workspace saved. No native engine execution claimed.


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Next</h2><p><a href="STUDENT.ipynb">Lab 07 native integration</a> → <a href="STUDENT.ipynb">Lab 08 serving</a>. Native prerequisites remain mandatory.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>التالي</h2><p><a href="STUDENT.ipynb">تكامل اللاب ٠٧</a> ثم <a href="STUDENT.ipynb">تقديم البيانات في اللاب ٠٨</a>. متطلبات التشغيل الأصلي لا تزال إلزامية.</p></td></tr></tbody></table>